In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                            classification_report, precision_score,
                            recall_score, f1_score)

# Load the dataset
file_path = "heart.csv"
data = pd.read_csv(file_path)

# Split the dataset into features and target
X = data.drop('target', axis=1)
y = data['target']

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Hyperparameter tuning using GridSearchCV
param_grid = {
    'max_depth': [3, 4, 5, 6],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2, 0.3],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'scale_pos_weight': [1, 2, 3]
}

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=3
)

# Fit the model with best parameters
grid_search.fit(X_train, y_train)

# Get the best parameters and model
best_params = grid_search.best_params_
print("\n🔥 Best Parameters:", best_params)
print("🔥 Best F1-Score:", grid_search.best_score_)

# Train final model with best parameters
xgb_best = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')
xgb_best.fit(X_train, y_train)

# Make predictions
y_pred = xgb_best.predict(X_val)

# Model evaluation
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print("\n🔥 **Model Performance:**")
print(f"✅ Accuracy: {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print(f"✅ F1-Score: {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], 
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Classification Report
print("\n🔥 **Classification Report:**")
print(classification_report(y_val, y_pred))

# Save the trained model
model_path = "xgboost_heart_model.pkl"
joblib.dump(xgb_best, model_path)
print(f"\n✅ Model saved at: {model_path}")

# Test the model with some sample cases
test_cases = np.array([
    [63, 1, 3, 145, 233, 1, 0, 150, 0, 2.3, 0, 0, 1],   # Positive case
    [67, 1, 2, 160, 286, 0, 1, 108, 1, 1.5, 1, 2, 3],   # Negative case
    [37, 1, 2, 130, 250, 0, 1, 187, 0, 3.5, 0, 0, 2],   # Positive case
    [56, 0, 1, 120, 240, 0, 0, 140, 1, 0.9, 0, 0, 1],   # Negative case
    [44, 1, 1, 130, 204, 0, 0, 172, 0, 0.0, 1, 0, 2]    # Positive case
])

# Convert test cases to DataFrame
X_test = pd.DataFrame(test_cases, columns=X.columns)

# Make predictions
predictions = xgb_best.predict(X_test)
probabilities = xgb_best.predict_proba(X_test)[:, 1]

# Display results

  
threshold = 0.5
print("\n🔥 **Model Test Results:**\n")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    label = "Positive (High Risk)" if prob >= threshold else "Negative (Low Risk)"
    print(f"Test Case {i + 1}: Prediction = {label}, Probability = {prob:.4f}")